# Lesson 4.4 — Midpoint Checkpoint: The Forward Path, Assembled

Assemble Perceive → Understand → Plan → Execute into one run and assert the full success predicate — perception to motion, no failure handling.

In [1]:
import numpy as np
from modules.module09.integration import (GreenhouseWorld, model_perception,
    understand, to_configuration, plan_reference, execute_reference,
    velocity_layer, geometric_jacobian, fk_xy, P2, T2)
checks = []
def forward_path(world, q_start, rng_seed=7):
    dets = model_perception(world, rng=np.random.default_rng(rng_seed))   # Perceive
    _, tgt = understand(dets, world)                                       # Understand
    if tgt is None: return None
    q_goal = to_configuration(tgt)                                         # (IK seam)
    if q_goal is None: return None
    layer = plan_reference(q_start, q_goal, rng=np.random.default_rng(0))  # Plan
    rec = execute_reference(layer)                                         # Execute
    arrived = fk_xy(P2, T2, rec['q'][-1])
    return dict(tgt=tgt, q_goal=q_goal, layer=layer, rec=rec, arrived=arrived)


In [2]:
world = GreenhouseWorld.demo_row(n=6, seed=1)
r = forward_path(world, world.q.copy())
print('committed:', r['tgt']['id'], '| validated:', r['layer']['validated'],
      '| reached:', r['rec']['reached'])
print('arrived at', np.round(r['arrived'],3), 'target', np.round(r['tgt']['xy'],3))
# the midpoint success predicate = conjunction of every stage's check
checks.append(r['tgt']['ripe'] and r['tgt']['reachable'])                       # Understand
checks.append(np.linalg.norm(fk_xy(P2,T2,r['q_goal']) - r['tgt']['xy']) < 1e-9) # IK seam
checks.append(r['layer']['validated'])                                          # Plan
checks.append(r['rec']['reached'])                                              # Execute
checks.append(np.linalg.norm(r['arrived'] - r['tgt']['xy']) < 1e-2)             # arrival
assert all(checks), f'FAILED: {checks}'
print(f'{sum(checks)}/{len(checks)} checks passed.')
print('MIDPOINT: forward path Perceive->Understand->Plan->Execute reaches the target.')
print('All checks passed.')


committed: F3 | validated: True | reached: True
arrived at [0.447 0.152] target [0.447 0.152]
5/5 checks passed.
MIDPOINT: forward path Perceive->Understand->Plan->Execute reaches the target.
All checks passed.
